# Eval Workflow

Notes for this chapter:

- 

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [19]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stream=False, output_config=None, output_format=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if output_config:
        params["output_config"] = output_config

    if output_format:
        params["output_format"] = output_format

    if stream:
        with client.messages.stream(**params) as response:
            for text in response.text_stream:
                print(text, end="", flush=True)
            message = response.get_final_message()
    else:
        if output_format:
            message = client.messages.parse(**params)
        else:
            message = client.messages.create(**params)

    # Responses can start with a thinking block, so find the text block
    return next(block.text for block in message.content if block.type == "text")

In [3]:
messages = []

add_user_message(messages, "Hello!")
answer = chat(messages)
add_assistant_message(messages, answer)

answer

'Hello! How can I help you today?'

## Exercises

# Building an eval pipeline for generating output files

Our prompt needs to assist users in writing three specific types of output for AWS use cases:

- Python code
- JSON configuration files
- Regular expressions

In [ ]:
import json
from pathlib import Path

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "Expect format of the output"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    output_config = {
          "format": {
              "type": "json_schema",
              "schema": {
                  "type": "object",
                  "properties": {
                      "tasks": {
                          "type": "array",
                          "items": {
                              "type": "object",
                              "properties": {
                                  "task": {"type": "string"},
                                  "format": {"type": "string"}
                              },
                              "required": ["task"],
                              "additionalProperties": False,
                          },
                      },
                  },
                  "required": ["tasks"],
                  "additionalProperties": False,
              },
          }
      }

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, output_config=output_config)
    return json.loads(text)

dataset = generate_dataset()
print(dataset)

path = Path("datasets/eval.json")
path.parent.mkdir(parents=True, exist_ok=True)

with open(path, 'w') as f:
    json.dump(dataset, f, indent=2)


{'tasks': [{'task': 'Write a Python function that takes an S3 bucket name and a prefix, and uses boto3 to return a list of all object keys under that prefix, handling pagination correctly.', 'format': 'python'}, {'task': "Create an IAM policy JSON document that grants read-only access (s3:GetObject and s3:ListBucket) to a bucket named 'my-app-logs' and all objects within it, and nothing else.", 'format': 'json'}, {'task': "Write a regular expression that matches a valid AWS EC2 instance ID, which is 'i-' followed by either 8 or 17 lowercase hexadecimal characters, and matches the entire string only.", 'format': 'regex'}]}


In [41]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task

* Respond only with {test_case["format"]}
* Do not add any comments or commentary or explanation
* Do not include any backtick code blocks, the code must be parsable as the output format

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset["tasks"]:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

with open("datasets/eval.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

[
  {
    "output": "import boto3\n\ndef list_s3_keys(bucket_name, prefix=\"\"):\n    s3_client = boto3.client(\"s3\")\n    paginator = s3_client.get_paginator(\"list_objects_v2\")\n    keys = []\n    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):\n        for obj in page.get(\"Contents\", []):\n            keys.append(obj[\"Key\"])\n    return keys",
    "test_case": {
      "task": "Write a Python function that takes an S3 bucket name and a prefix, and uses boto3 to return a list of all object keys under that prefix, handling pagination correctly.",
      "format": "python"
    },
    "score": 10
  },
  {
    "output": "{\"Version\":\"2012-10-17\",\"Statement\":[{\"Sid\":\"ListBucket\",\"Effect\":\"Allow\",\"Action\":\"s3:ListBucket\",\"Resource\":\"arn:aws:s3:::my-app-logs\"},{\"Sid\":\"GetObjects\",\"Effect\":\"Allow\",\"Action\":\"s3:GetObject\",\"Resource\":\"arn:aws:s3:::my-app-logs/*\"}]}",
    "test_case": {
      "task": "Create an IAM policy JSON document

## Model based grading

Using another model to grade the answers 

In [23]:
from pydantic import BaseModel

class Grading(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: int


def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)
    text = chat(messages, output_format=Grading)
    print(text)
    return json.loads(text)

sum = 0

for result in results:
    grading = grade_by_model(result["test_case"]["task"], result["output"])
    sum += grading["score"]

print(f"Average is {sum/len(results)}")


{"strengths":["Correctly uses boto3's list_objects_v2 paginator, so pagination is handled reliably without manual continuation token logic.","Safely handles empty results with page.get(\"Contents\", []), avoiding KeyError on empty prefixes or buckets.","Optional injectable client parameter makes the function easy to test, and the notes accurately cover prefix semantics, versioning, permissions, and scale."],"weaknesses":["Wraps ClientError in RuntimeError, which contradicts the docstring (it says ClientError is raised) and discards the typed exception that callers would normally want to catch, such as NoSuchBucket.","Signature makes prefix optional, whereas the task specifies it as an input; this is minor but deviates slightly. There are also no unit tests or moto-based example.","Some notes are slightly imprecise or tangential, e.g. the claim that console-created folder keys always have size 0 and the 'current versions' remark, which is fine but adds length without code to support it.

# Code based grading

Where appropriate, use code-based tests to grade the output. For example if generating json check the shape, and that it is valid json

In [42]:
import ast
import re


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text)
        return 10
    except re.error as e:
        print(e.msg)
        return 0

def grade_by_model(format, output):

    match format:
        case "python": 
            return validate_python(output)
        case "json": 
            return validate_json(output)
        case "regex": 
            return validate_regex(output)

    return 0


sum = 0

for result in results:
    format = result["test_case"]["format"]
    grading = grade_by_model(format, result["output"])
    print(f"Result for {format} is {grading}")
    sum += grading

print(sum / len(results))

Result for python is 10
Result for json is 10
Result for regex is 10
10.0
